# Session 19 lab: forecast 2025, beat the baseline, build a Pipeline

**AI and Data Science Professional** | KLAi Professional Learning | Project milestone 3

You forecast malaria cases for 12 districts in 2025 using only what was known before 2025. You build the seasonal naive baseline, add lag features, score with MAE, RMSE and MAPE, pack the work in a scikit-learn Pipeline, and write the three sentences that go to a district manager.

**How to use this notebook**

1. Run the cells in order from the top (Shift and Enter).
2. Every step has the same rhythm: **PREDICT**, run, **READ**, then **CHANGE** one thing.
3. Cells that say `answer = None` are yours to fill in. Replace `None` with your code. Then run the check cell under it.
4. The data is fictional and generated. Cases are counts of malaria cases per district per month.
5. If you get an error, read the last line first.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

---
## Step 1: Load the clinic file and sort it

A time series must be in time order inside each district. Each record is one district in one month.

In [ ]:
try:
    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.pipeline import make_pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LinearRegression

    os.makedirs("keep", exist_ok=True)   # the folder where you save what you keep
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

**PREDICT first.** How many records will the file have? (12 districts, 60 months.) And how many blank values will total_visits have?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    answer = None  # your code here: read clinic_visits.csv with month as a date, then sort by district and month
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    clinic = answer
    n_rows = len(clinic)
    n_blank = int(clinic.total_visits.isna().sum())
    print(n_rows, n_blank)
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")
try:
    check("Step 1a: records in the file", n_rows, 720)
except Exception:
    check('Step 1a: records in the file', float("nan"), 0, 0)
try:
    check("Step 1b: blank total_visits", n_blank, 18)
except Exception:
    check('Step 1b: blank total_visits', float("nan"), 0, 0)

**READ the output.** Which variable are we going to forecast, and what is the unit of one record?

> My answer: 

---
## Step 2: The baseline model: seasonal naive

Forecast every month of 2025 with the same month of 2024. Any model you build later must beat this.

**PREDICT first.** Will the baseline miss by more in Gulu (about 3,700 cases a month) or in Kampala (about 650)? Guess the size of each miss.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    gulu = clinic[clinic.district == 'Gulu'].set_index('month').malaria_cases
    actual = gulu.loc['2025'].values       # what happened in 2025
    last_year = gulu.loc['2024'].values    # the forecast: the same month last year
    print(len(actual), len(last_year))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    answer = None  # your code here: the average of the absolute gaps between actual and last_year (this is MAE)
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    gulu_mae = answer
    print("Gulu baseline MAE:", round(float(gulu_mae), 1), "cases")
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")
try:
    check("Step 2a: Gulu baseline MAE", gulu_mae, 980.1667, 0.1)
except Exception:
    check('Step 2a: Gulu baseline MAE', float("nan"), 0, 0)

**CHANGE one thing.** Run the same steps for Kampala: change the district name in the first cell of this step and run the cells again. Write down the Kampala MAE, then compare it with the mean cases of each district.

> What I saw: 

In [ ]:
try:
    kampala = clinic[clinic.district == 'Kampala'].set_index('month').malaria_cases
    kampala_mae = abs(kampala.loc['2025'].values - kampala.loc['2024'].values).mean()
    print('Kampala baseline MAE:', round(float(kampala_mae), 1))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")
try:
    check("Step 2b: Kampala baseline MAE", kampala_mae, 120.7500, 0.1)
except Exception:
    check('Step 2b: Kampala baseline MAE', float("nan"), 0, 0)

---
## Step 3: Seasonality and the delayed effect of rain

Add every district together, look at the calendar month with the most cases, then test whether rain works with a delay.

**PREDICT first.** Which calendar month (1 to 12) has the most cases over the five years? Which month do you guess is the trough?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    answer = None  # your code here: group by the calendar month number, add the cases, and take the month with the largest total (idxmax)
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    peak_month = answer
    print("Peak month:", peak_month)
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")
try:
    check("Step 3a: peak calendar month", peak_month, 12)
except Exception:
    check('Step 3a: peak calendar month', float("nan"), 0, 0)

**PREDICT first.** Last month's rain: will its correlation with this month's cases be near 0 or clearly above 0? And this month's rain?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    g = clinic.groupby('district')      # every shift stays inside one district
    clinic['cases_lag1'] = g.malaria_cases.shift(1)
    clinic['rain_lag1'] = g.rainfall_mm.shift(1)
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    answer = None  # your code here: the cases of the same month last year, shifted inside each district: shift(12)
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    clinic['cases_lag12'] = answer
    features = ['cases_lag1', 'cases_lag12', 'rain_lag1']
    clinic[clinic.district == 'Gulu'][['month', 'malaria_cases'] + features].iloc[0:15]
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

**READ the output.** Look at January 2022 for Gulu. Where does its cases_lag12 value come from? Why are the first rows blank?

> My answer: 

In [ ]:
try:
    answer = None  # your code here: keep only the records that have all three lag features
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    ready = answer
    r_now = ready.rainfall_mm.corr(ready.malaria_cases)
    r_lag = ready.rain_lag1.corr(ready.malaria_cases)
    print(len(ready), round(r_now, 2), round(r_lag, 2))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")
try:
    check("Step 3b: records with every lag feature", len(ready), 576)
except Exception:
    check('Step 3b: records with every lag feature', float("nan"), 0, 0)
try:
    check("Step 3c: correlation with this month's rain", r_now, 0.0186, 0.01)
except Exception:
    check("Step 3c: correlation with this month's rain", float("nan"), 0, 0)
try:
    check("Step 3d: correlation with last month's rain", r_lag, 0.3725, 0.01)
except Exception:
    check("Step 3d: correlation with last month's rain", float("nan"), 0, 0)

**READ the output.** What do the two correlations say about the effect of rain? Why would a model that sees only this month's rain miss it?

> My answer: 

---
## Step 4: Split by time

Train on 2022 to 2024. Test on 2025. Never split a time series at random.

**PREDICT first.** How many records will the training set have, and how many the test set? (Remember 12 districts.)

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    answer = None  # your code here: the records with month before 1 January 2025
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    train = answer
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")
try:
    check("Step 4a: training records", len(train), 432)
except Exception:
    check('Step 4a: training records', float("nan"), 0, 0)

In [ ]:
try:
    answer = None  # your code here: the records with month from 1 January 2025
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    test = answer
    print(len(train), len(test))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")
try:
    check("Step 4b: test records", len(test), 144)
except Exception:
    check('Step 4b: test records', float("nan"), 0, 0)
try:
    check("Step 4c: training ends before the test starts", float(train.month.max() < test.month.min()), 1)
except Exception:
    check('Step 4c: training ends before the test starts', float("nan"), 0, 0)

**CHANGE one thing.** Change the cut date to 1 January 2024 in both cells and look at the new counts. Then put it back to 2025 and run the cells again.

> What I saw: 

---
## Step 5: Fit the lag model and score it three ways

A linear regression fitted with numpy on the training set, then scored on the test set against the baseline. MAE, RMSE and MAPE all use the actual and forecast values.

In [ ]:
try:
    X_train = np.c_[np.ones(len(train)), train[features]]   # a column of ones for the intercept
    y_train = train.malaria_cases
    X_test = np.c_[np.ones(len(test)), test[features]]
    y_test = test.malaria_cases
    baseline_pred = test.cases_lag12   # seasonal naive forecast
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    answer = None  # your code here: least squares: np.linalg.lstsq(X_train, y_train, rcond=None)[0] gives the intercept and one weight per feature
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    coef = answer
    y_pred = X_test @ coef
    print(coef.round(2))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

**PREDICT first.** Will the model's MAE be smaller than the baseline's? By how many cases, roughly?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    answer = None  # your code here: the mean absolute error of the baseline on the test set
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    naive_mae = answer
    print("baseline MAE:", round(float(naive_mae), 1))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")
try:
    check("Step 5a: baseline MAE", naive_mae, 460.6667, 0.1)
except Exception:
    check('Step 5a: baseline MAE', float("nan"), 0, 0)

In [ ]:
try:
    answer = None  # your code here: the mean absolute error of the model on the test set
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    model_mae = answer
    print("model MAE:", round(float(model_mae), 1))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")
try:
    check("Step 5b: model MAE", model_mae, 337.9681, 0.1)
except Exception:
    check('Step 5b: model MAE', float("nan"), 0, 0)

In [ ]:
try:
    answer = None  # your code here: RMSE: square each miss, take the mean, then the square root
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    model_rmse = answer
    print("model RMSE:", round(float(model_rmse), 1))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")
try:
    check("Step 5c: model RMSE", model_rmse, 490.8514, 0.1)
except Exception:
    check('Step 5c: model RMSE', float("nan"), 0, 0)

In [ ]:
try:
    answer = None  # your code here: MAPE: each absolute miss divided by the actual value, then the mean, times 100
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    model_mape = answer
    print("model MAPE:", round(float(model_mape), 1), "percent")
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")
try:
    check("Step 5d: model MAPE", model_mape, 16.3295, 0.05)
except Exception:
    check('Step 5d: model MAPE', float("nan"), 0, 0)

**READ the output.** The RMSE is larger than the MAE. What does that tell you about the size of the biggest misses? In which unit is each of the three scores?

> My answer: 

**CHANGE one thing.** Refit the model with only features = ["cases_lag1"] (rebuild X_train and X_test first). Is it still better than the baseline? The slides give the answer, but check it yourself.

> What I saw: 

---
## Step 6: Pack the job into a Pipeline

A Pipeline chains preprocessing and the model, so the training and the test records go through the same steps, learned from the training records only.

**PREDICT first.** The Pipeline fills blanks, standardises and fits the same linear regression. Will its MAE be the same as your numpy model, a little different, or very different?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    answer = None  # your code here: make_pipeline with SimpleImputer(strategy="median"), StandardScaler() and LinearRegression()
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    pipe = answer
    pipe.fit(train[features], train.malaria_cases)
    y_pipe = pipe.predict(test[features])
    pipe_mae = abs(y_test - y_pipe).mean()
    print("Pipeline MAE:", round(float(pipe_mae), 1))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")
try:
    check("Step 6a: Pipeline MAE equals the numpy MAE", pipe_mae, 337.9681, 0.1)
except Exception:
    check('Step 6a: Pipeline MAE equals the numpy MAE', float("nan"), 0, 0)

**READ the output.** Why does standardising the variables not change the forecast of a linear regression?

> My answer: 

In [ ]:
try:
    # Districts differ a lot in size. Add the district as indicator variables (one-hot encoding with pandas; drop the first district).
    dummies = pd.get_dummies(ready.district, dtype=float).iloc[:, 1:]
    ready_d = pd.concat([ready, dummies], axis=1)
    features_d = features + list(dummies.columns)
    train_d = ready_d[ready_d.month < '2025-01-01']
    test_d = ready_d[ready_d.month >= '2025-01-01']
    print(len(features_d), 'variables')
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    answer = None  # your code here: fit the same kind of Pipeline on train_d[features_d] and predict test_d[features_d]
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    y_pipe_d = answer
    pipe_d_mae = abs(test_d.malaria_cases - y_pipe_d).mean()
    print("Pipeline with district indicator variables, MAE:", round(float(pipe_d_mae), 1))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")
try:
    check("Step 6b: Pipeline MAE with district indicator variables", pipe_d_mae, 217.0848, 0.1)
except Exception:
    check('Step 6b: Pipeline MAE with district indicator variables', float("nan"), 0, 0)

**CHANGE one thing.** In scikit-learn the encoding goes inside the Pipeline with a ColumnTransformer and a OneHotEncoder. Read the documentation page of OneHotEncoder (drop="first") and write in a text cell how you would build it. You do not need to run it today.

> What I saw: 

---
## Step 7: See the lag trap with your own eyes

Build the same lags without groupby. Python gives no error. Read what it does to the score.

**PREDICT first.** Without groupby, will the test MAE go up, go down, or stay the same? Why?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
try:
    bad = clinic[['district', 'month', 'malaria_cases', 'rainfall_mm']].copy()
    bad['cases_lag1'] = bad.malaria_cases.shift(1)     # the mistake: no groupby
    bad['cases_lag12'] = bad.malaria_cases.shift(12)
    bad['rain_lag1'] = bad.rainfall_mm.shift(1)
    bad = bad.dropna()
    train_bad = bad[bad.month < '2025-01-01']
    test_bad = bad[bad.month >= '2025-01-01']
    coef_bad = np.linalg.lstsq(np.c_[np.ones(len(train_bad)), train_bad[features]], train_bad.malaria_cases, rcond=None)[0]
    y_pred_bad = np.c_[np.ones(len(test_bad)), test_bad[features]] @ coef_bad
    print(len(train_bad), 'training records instead of', len(train))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    answer = None  # your code here: the mean absolute error of the model built on the wrong lags
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    bad_mae = answer
    print("MAE with the wrong lags:", round(float(bad_mae), 1), "against", round(float(model_mae), 1))
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")
try:
    check("Step 7: MAE with the wrong lags", bad_mae, 366.3342, 0.1)
except Exception:
    check('Step 7: MAE with the wrong lags', float("nan"), 0, 0)

**READ the output.** The wrong version keeps 132 extra training records. Where do their lag values come from?

> My answer: 

---
## Step 8: Your Project milestone 3 results file

Collect the three models in one table and save it. This file and the three sentences below are the evidence you keep.

In [ ]:
try:
    def scores(y, p):
        mae = abs(y - p).mean()
        rmse = np.sqrt(((y - p) ** 2).mean())
        mape = (abs(y - p) / y).mean() * 100
        return round(float(mae), 1), round(float(rmse), 1), round(float(mape), 1)

    rows = {"Seasonal naive baseline": scores(y_test, baseline_pred),
            "Lag model (numpy)": scores(y_test, y_pred),
            "Pipeline with district indicator variables": scores(test_d.malaria_cases, y_pipe_d)}
    results = pd.DataFrame(rows, index=["MAE (cases)", "RMSE (cases)", "MAPE (percent)"]).T
    results.to_csv("keep/session19_forecast_results.csv")
    results
except Exception as e:
    print("Finish the cells above first. (" + type(e).__name__ + ")")

In [ ]:
try:
    check("Step 8: the results table has three models", len(results), 3)
except Exception:
    check('Step 8: the results table has three models', float("nan"), 0, 0)

In [ ]:
show_checks()

## Your three sentences (Project milestone 3)

Double-click this cell and write three sentences for a district manager. Use your own numbers.

1. The forecast horizon and the average miss of the model, in cases (and as a share of a typical month).
2. Whether it beats the seasonal naive baseline, and by how much.
3. One limit: for example one test year only, districts where the baseline wins, or a horizon longer than one month.

> My three sentences: 